# 06 · Reducers (how updates are merged)

**Goal:** by the end you can choose how each state key is updated: overwritten (the default), appended to with
`operator.add`, or combined by a reducer function you write yourself.

## The idea in plain English

Every time a node returns an update, LangGraph has to merge it into the state. Until now each key followed one rule:
**the new value replaces the old one**. That's right for `result`, but wrong for a calculation *history*: we want
each new line added to the end, not the whole list replaced.

A **reducer** is a function that decides how a key is merged. It takes the old value and the new value and returns
the combined value. You attach a reducer to a key with `Annotated[type, reducer]` in the state schema. Keys without
a reducer keep the default rule (replace).

**Analogy:** a whiteboard versus a logbook. On a whiteboard you erase the old number and write the new one
(no reducer). In a logbook you never erase: each entry goes on a new line at the bottom (`operator.add`).

## Picture

Two nodes run one after the other. Both write to `last` (no reducer) and `history` (with a reducer).

```mermaid
flowchart LR
    S([START]) --> first --> second --> E([END])
    first -. "last = 'first'<br/>history + ['first']" .-> state[(state)]
    second -. "last = 'second'<br/>history + ['second']" .-> state
```

| Key | Reducer | After `first` | After `second` |
|---|---|---|---|
| `last` | none (replace) | `'first'` | `'second'` |
| `history` | `operator.add` (append) | `['start', 'first']` | `['start', 'first', 'second']` |

## Step 1: What `operator.add` does

`operator.add(x, y)` is the function version of `x + y`. For two lists, `+` joins them into one list. That's exactly
the merge we want for `history`: old list + new items. It takes the old value and the new value, so it can be used
as a reducer as it is.

In [1]:
import operator

print(operator.add(3, 4))
print(operator.add(["start"], ["first"]))

7
['start', 'first']


## Step 2: A state with and without a reducer

**`Annotated`** lets you attach extra information to a type hint. `Annotated[list[str], operator.add]` still means
"a list of strings", plus a note LangGraph reads: "merge this key with `operator.add`".

`last` is a plain `str`, so it gets the default rule. `history` gets the reducer.

In [2]:
from typing import Annotated, TypedDict

from langgraph.graph import StateGraph, START, END


class DemoState(TypedDict):
    last: str  # no reducer: each update replaces the old value
    history: Annotated[list[str], operator.add]  # reducer: each update is appended

## Step 3: Two nodes write to both keys

Each node returns the **new** part only: a new `last`, and a list holding the **one** new history line. It does not
return the old history plus the new line. The reducer does the joining.

In [3]:
def first(state: DemoState) -> dict:
    return {"last": "first", "history": ["first"]}


def second(state: DemoState) -> dict:
    return {"last": "second", "history": ["second"]}


builder = StateGraph(DemoState)
builder.add_node("first", first)
builder.add_node("second", second)
builder.add_edge(START, "first")
builder.add_edge("first", "second")
builder.add_edge("second", END)
demo = builder.compile()
print(demo.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	first(first)
	second(second)
	__end__([<p>__end__</p>]):::last
	__start__ --> first;
	first --> second;
	second --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 4: Before and after, side by side

We start both keys with the value `"start"` and run the graph. `last` ends up holding only the final write.
`history` kept everything: the input value, then one line from each node, in order.

In [4]:
before = {"last": "start", "history": ["start"]}
after = demo.invoke(before)

print(f"{'key':8} {'before':20} after")
for key in ["last", "history"]:
    print(f"{key:8} {str(before[key]):20} {after[key]}")

key      before               after
last     start                second
history  ['start']            ['start', 'first', 'second']


The input counts too. The reducer joined the input `['start']` with the nodes' updates. If we don't pass a history,
the list starts empty.

In [5]:
print(demo.invoke({}))

{'last': 'second', 'history': ['first', 'second']}


## Step 5: A calculator that keeps a history

Now a real use. The calculator from lesson 05, cut down to two operations so it stays short. Every node adds one
line to `history`, describing what it did. At the end, `history` is a step-by-step log of the run.

In [6]:
from typing import Literal


class CalcState(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str
    history: Annotated[list[str], operator.add]

The nodes. Each one returns its normal update **plus** a one-item list for `history`, describing what it did.

In [7]:
def parse(state: CalcState) -> dict:
    left, op, right = state["expression"].split()
    a, b = float(left), float(right)
    return {"a": a, "op": op, "b": b, "history": [f"parse: a={a:g}, op={op}, b={b:g}"]}


def add(state: CalcState) -> dict:
    result = state["a"] + state["b"]
    return {"result": result, "history": [f"add: {result:g}"]}


def multiply(state: CalcState) -> dict:
    result = state["a"] * state["b"]
    return {"result": result, "history": [f"multiply: {result:g}"]}

`format_answer` also logs a line. `route` is the lesson 05 router, cut down to two operations.

In [8]:
def format_answer(state: CalcState) -> dict:
    answer = f"{state['expression']} = {state['result']:g}"
    return {"answer": answer, "history": [f"format: {answer}"]}


def route(state: CalcState) -> Literal["add", "multiply"]:
    return "add" if state["op"] == "+" else "multiply"

The wiring is the same as in lesson 05. The reducer needs no wiring of its own: it's part of the state schema.

In [9]:
builder = StateGraph(CalcState)
builder.add_node("parse", parse)
builder.add_node("add", add)
builder.add_node("multiply", multiply)
builder.add_node("format", format_answer)
builder.add_edge(START, "parse")
builder.add_conditional_edges("parse", route)
builder.add_edge("add", "format")
builder.add_edge("multiply", "format")
builder.add_edge("format", END)
calculator = builder.compile()
print(calculator.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	add(add)
	multiply(multiply)
	format(format)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	add --> format;
	multiply --> format;
	parse -.-> add;
	parse -.-> multiply;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 6: Run it and read the history

In [10]:
run1 = calculator.invoke({"expression": "3 + 4"})
print(run1["answer"])
for line in run1["history"]:
    print("  ", line)

3 + 4 = 7
   parse: a=3, op=+, b=4
   add: 7
   format: 3 + 4 = 7


Each `invoke` starts fresh, so a second run has its own history. If we pass the first run's history in as input, the
reducer appends the new lines after it. Lesson 13 shows how LangGraph can remember this for you between runs.

In [11]:
run2 = calculator.invoke({"expression": "6 * 7", "history": run1["history"]})
for line in run2["history"]:
    print("  ", line)

   parse: a=3, op=+, b=4
   add: 7
   format: 3 + 4 = 7
   parse: a=6, op=*, b=7
   multiply: 42
   format: 6 * 7 = 42


## Step 7: A custom reducer

A reducer is any function with two inputs, the old value and the new value, that returns the merged value.
`keep_last_3` appends, then keeps only the three newest items. Like a node, it's just a function, so we test it
directly first.

In [12]:
def keep_last_3(old: list[str], new: list[str]) -> list[str]:
    """Append the new items, then keep only the 3 newest."""
    return (old + new)[-3:]


print(keep_last_3(["a", "b"], ["c"]))
print(keep_last_3(["a", "b", "c"], ["d", "e"]))

['a', 'b', 'c']
['c', 'd', 'e']


Now we use it in a schema. To keep the demo small, two tiny nodes each log one line, and we start with three old
lines. Only the three newest lines survive.

Why not reuse the calculator's nodes? LangGraph reads a node's type hint (`state: CalcState`) to learn its state
keys. The calculator nodes would bring along `CalcState`'s `operator.add` reducer for `history`, which clashes with
`keep_last_3`: `ValueError: Channel 'history' already exists with a different type`.

In [13]:
class ShortHistoryState(TypedDict):
    history: Annotated[list[str], keep_last_3]


def log_add(state: ShortHistoryState) -> dict:
    return {"history": ["add: 7"]}


def log_format(state: ShortHistoryState) -> dict:
    return {"history": ["format: 3 + 4 = 7"]}

The graph runs `log_add`, then `log_format`. We start it with three old lines, so five lines reach the reducer in total.

In [14]:
builder = StateGraph(ShortHistoryState)
builder.add_node("log_add", log_add)
builder.add_node("log_format", log_format)
builder.add_edge(START, "log_add")
builder.add_edge("log_add", "log_format")
builder.add_edge("log_format", END)
short = builder.compile()

out = short.invoke({"history": ["old 1", "old 2", "old 3"]})
print(out["history"])

['old 3', 'add: 7', 'format: 3 + 4 = 7']


## Step 8 (preview): `add_messages` and `MessagesState`

Chat programs keep a list of **messages** (what the user said, what the AI replied). LangGraph has a ready-made
reducer for that, **`add_messages`**. It appends new messages like `operator.add`, with one extra trick: a message
with the same `id` as an existing one *replaces* it instead of being added twice.

**`MessagesState`** is a ready-made state with a single key, `messages`, that already uses `add_messages`. You'll use
it in lessons 18–19. For now we just call the reducer directly. (Each message has a type, such as `HumanMessage` or
`AIMessage`, and a `content` holding its text.)

In [15]:
from langchain.messages import AIMessage, HumanMessage
from langgraph.graph import MessagesState
from langgraph.graph.message import add_messages

old = [HumanMessage("What is 3 + 4?", id="m1")]

appended = add_messages(old, [AIMessage("7", id="m2")])
print("append: ", [(type(m).__name__, m.content) for m in appended])

replaced = add_messages(old, [HumanMessage("What is 3 * 4?", id="m1")])  # same id
print("replace:", [(type(m).__name__, m.content) for m in replaced])

print("MessagesState keys:", list(MessagesState.__annotations__))

append:  [('HumanMessage', 'What is 3 + 4?'), ('AIMessage', '7')]
replace: [('HumanMessage', 'What is 3 * 4?')]
MessagesState keys: ['messages']


## What just happened

- **Step 1** showed `operator.add(["start"], ["first"])` gives `['start', 'first']`, which makes it a good reducer
  for lists.
- **Step 4** ran two nodes over the same two keys. `last` went `'start'` → `'second'` (replaced), while `history`
  went `['start']` → `['start', 'first', 'second']` (appended).
- **Step 6** gave the calculator a `history` that logged every node: `parse: a=3, op=+, b=4`, `add: 7`,
  `format: 3 + 4 = 7`. Passing it into a second run made the new lines follow the old ones.
- **Step 7** wrote `keep_last_3`. Starting from three old lines plus two new ones, only the newest three were kept:
  `['old 3', 'add: 7', 'format: 3 + 4 = 7']`.
- **Step 8** used `add_messages`: a new `id` was appended, and the same `id` replaced the old message.

## Common mistakes

1. **Returning a string instead of a list.** With `operator.add`, `{"history": "add: 7"}` tries to join a list and a
   string:
   `TypeError: can only concatenate list (not "str") to list`.
   Fix: wrap the new item in a list, as in `{"history": ["add: 7"]}`.
2. **Returning the whole history plus the new line.** `{"history": state["history"] + ["add: 7"]}` gives **no
   error**, but the reducer appends the old lines *again*: `['a', 'b']` becomes `['a', 'b', 'a', 'b', 'add: 7']`.
   Fix: return only the new items. The reducer adds the old ones.
3. **Forgetting `Annotated`.** Writing `history: list[str]` gives **no error** either. Each node's list simply
   replaces the previous one, and you end up with only the last line.
   Fix: `history: Annotated[list[str], operator.add]`.

## Try it

1. Add `steps: Annotated[int, operator.add]` to `DemoState`. Make `first` and `second` each also return
   `{"steps": 1}`. Run `demo.invoke({"steps": 0})`. What is `steps` at the end, and why?
2. Write a reducer `no_duplicates(old, new)` that appends only the items not already in the list. Test it directly:
   `no_duplicates(["add: 7"], ["add: 7", "multiply: 42"])` should give `['add: 7', 'multiply: 42']`.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Reducer**: a function `(old, new) -> merged` that decides how updates to one state key are combined.
- **`Annotated`**: attaches extra information to a type hint. `Annotated[list[str], operator.add]` attaches a reducer.
- **`operator.add`**: the function version of `+`. As a reducer it appends lists (or adds numbers).
- **`add_messages`**: LangGraph's reducer for message lists. It appends, but replaces a message with the same `id`.
- **`MessagesState`**: a ready-made state with one key, `messages`, that uses `add_messages`.
- **Message**: one turn of a chat, such as a `HumanMessage` or `AIMessage`, with its text in `content`.